# SpaNCy-Shift Stage 2 — matched-noise controls, seeds 1–2

Lean seed repeat of the GNN part of `spancy_shift_ddpm_avg.ipynb` (no DDPM). That run showed, at
seed 0, that Stage 1 + Gaussian noise **matched to the Stage 2 correction** reaches most of Stage 2's
kBET, but costs silhouette and 1D shape, while Stage 2 does not. This notebook repeats the biology
side at seeds 1 and 2 so the claim rests on 3 seeds.

Per seed: train the published config (`w_adv=0.3`, α=0.6, 10 epochs), then build
- **S1+noise**: Stage 1 + zero-mean Gaussian noise, SD matched per batch and marker to that seed's correction
- **S1+shuffled**: Stage 1 + that seed's own corrections, shuffled across cells of the same batch

and measure kBET, per-sample silhouette and 1D shape for all three. §9 combines seeds 1–2 with the
seed-0 numbers (hard-coded from the ddpm_avg run) into paired 3-seed results.

Positive population is skipped: it was 11/20 for every arm, so it cannot tell them apart.

**To run:** Runtime → Run all (GPU). Upload **`spancy_shift.py`** when §0 asks. ~2 × 8 min training,
then kBET on 6 layers. **Paste back §9 only** (plus any error).

## 0. Colab Setup

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload spancy_shift.py (requested only if missing).
from google.colab import files
import os

NEEDED = ['spancy_shift.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()

for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:24s} {os.path.getsize(f):>10,} bytes')

In [ ]:
import gc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift
from spancy_shift import (
    load_adata, detect_bimodal_markers, find_best_sample_per_marker,
    train, normalize_adata,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load data, bimodal markers, reference samples

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: DDPM scores the full joint distribution — no special handling required.")
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)
uni_markers = [m for m in marker_names if m not in bimodal_names]
uni_idx = [marker_names.index(m) for m in uni_markers]
print(f'{len(uni_markers)} unimodal markers, bimodal: {bimodal_names}')

## 3. Train + controls, per seed

Same training call and inference seed as the ddpm_avg notebook. Each seed's controls are matched to
that seed's own correction (log1p space), so every comparison in §9 is paired within a seed.

In [ ]:
SEEDS = [1, 2]        # seed 0 already measured in spancy_shift_ddpm_avg.ipynb (hard-coded in §9)
HYBRID_ALPHA = 0.6
K_NEIGHBORS = 15


def add_controls(delta, key_gauss, key_perm, rng):
    """Stage 1 + Gaussian noise (SD matched per batch and marker) / + shuffled delta."""
    g = base_log.copy()
    p = base_log.copy()
    for b in np.unique(batches_arr):
        ii = np.where(batches_arr == b)[0]
        db = delta[ii]
        g[ii] += rng.standard_normal(db.shape).astype(np.float32) * db.std(axis=0)
        p[ii] += db[rng.permutation(len(ii))]
    adata_norm.layers[key_gauss] = np.clip(np.expm1(g), 0, None).astype(np.float32)
    adata_norm.layers[key_perm] = np.clip(np.expm1(p), 0, None).astype(np.float32)


adata_norm = None
for seed in SEEDS:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    m_s, sc_s, ref_s, h_s = train(
        adata, n_epochs=10, lr=1e-3, device_str=DEVICE, warmup_epochs=2,
        n_per_batch=512, k_neighbors=K_NEIGHBORS, hybrid_alpha=HYBRID_ALPHA,
        temperature=0.07, w_recon=0.1, w_contrast=0.5, w_adv=0.3, w_mmd=1.0,
        mmd_samples=256, grl_max=1.0, bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
        ref_sample_per_marker=ref_sample_per_marker,
    )
    torch.manual_seed(100)
    torch.cuda.manual_seed_all(100)
    out = normalize_adata(
        adata, m_s, sc_s, ref_s, hybrid_alpha=HYBRID_ALPHA, k_neighbors=K_NEIGHBORS,
        inference_batch_size=2048, device_str=DEVICE,
        layer_name='normalized', keep_base_layer=True,
    )
    g = np.asarray(out.layers['normalized'], dtype=np.float32)
    if adata_norm is None:
        adata_norm = out                      # keeps 'normalized_base' (Stage 1)
        del adata_norm.layers['normalized']
        X_base = np.asarray(adata_norm.layers['normalized_base'])
        base_log = np.log1p(np.clip(X_base, 0, None)).astype(np.float32)
        batches_arr = adata_norm.obs['batch_id'].astype(str).values
    else:
        s1_diff = np.abs(np.asarray(out.layers['normalized_base']) - X_base).max()
        print(f'Stage 1 check across seeds: max |diff| = {s1_diff:.2e}  (expect 0)')
        del out
    adata_norm.layers[f'gnn_s{seed}'] = g

    delta = np.log1p(np.clip(g, 0, None)) - base_log
    add_controls(delta, f'gctrl_gauss_s{seed}', f'gctrl_perm_s{seed}', np.random.default_rng(seed))
    print(f'GNN seed {seed}: mean |delta| (unimodal) = {np.abs(delta[:, uni_idx]).mean():.4f}')
    del m_s, sc_s, ref_s, h_s, delta, g
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ABL_LAYERS = [('normalized_base', 'Stage 1')]
for s in SEEDS:
    ABL_LAYERS += [(f'gnn_s{s}', f'GNN s{s}'),
                   (f'gctrl_gauss_s{s}', f'GNN-s{s} S1+noise'),
                   (f'gctrl_perm_s{s}', f'GNN-s{s} S1+shuffled')]
ABL_LABEL = dict(ABL_LAYERS)
BIO_LAYERS = ABL_LAYERS
print(f'\nLayers: {list(adata_norm.layers.keys())}')

## 6b. 1D shape (unimodal markers)

In [ ]:
# ============================================================================
# 6b. SHAPE-PRESERVATION DIAGNOSTIC  (location-invariant)  — per layer
# ----------------------------------------------------------------------------
# Positive-pop Δ (fraction over a threshold) and silhouette (3-blob 20D
# separation) are both SHAPE-BLIND: you can flatten a sharp peak into a broad
# smear and still pass both. This cell measures whether each marker's 1D
# marginal SHAPE survives, INDEPENDENT of any location shift Stage 1 / GNN apply.
#
# Per (marker, sample), in log1p space, normalized vs raw (1.0 = shape preserved):
#   peak_ratio = peak density(norm) / peak density(raw)   <1 => mode flattened
#   var_ratio  = var(norm) / var(raw)                     >1 => broadened
#   iqr_ratio  = IQR(norm) / IQR(raw)                     >1 => broadened (robust)
#
# peak_ratio uses SHARED bin edges (union range) so it is binning-fair.
# Stage 1 is a pure translation -> should sit ~1.0 everywhere (control).
# NOTE: bimodal markers (ECAD etc.) are masked from the GNN's MMD loss, so their
# delta stays ~0 and they trivially score ~1.0 for every arm — they prove nothing
# about the ablation. Judge the arms on the unimodal markers the GNN corrects.
#
# ABLATION READING: if an arm drops a loss and peak/var stay ~1.0 while kBET holds,
# that loss was not buying shape safety.
# ============================================================================
from scipy.stats import iqr as _iqr

SHAPE_LAYERS = globals().get('BIO_LAYERS',
                             [('normalized_base', 'Stage 1'), ('normalized', 'full')])
SHAPE_LABELS = [label for _, label in SHAPE_LAYERS]
S1_LABEL = SHAPE_LABELS[0]                                  # control (pure shift)
MAIN_LABEL = SHAPE_LABELS[1] if len(SHAPE_LABELS) > 1 else S1_LABEL
N_SHAPE_BINS = 80
MIN_CELLS    = 50

# --- precompute ONCE ---
sample_ids_shape = adata_norm.obs['sample_id'].values
sample_unique    = np.unique(sample_ids_shape)
sample_idx       = {s: np.where(sample_ids_shape == s)[0] for s in sample_unique}
sample_idx       = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log        = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


rows = []
for layer_key, layer_label in SHAPE_LAYERS:
    X_layer_log = np.log1p(np.clip(np.asarray(adata_norm.layers[layer_key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        raw_col  = X_raw_log[:, mi]
        norm_col = X_layer_log[:, mi]
        peaks, vrs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(raw_col[idx], norm_col[idx])
            if np.isfinite(p):  peaks.append(p)
            if np.isfinite(v):  vrs.append(v)
            if np.isfinite(ir): irs.append(ir)
        rows.append({
            'method': layer_label, 'marker': mname,
            'peak_ratio': np.mean(peaks) if peaks else np.nan,
            'var_ratio':  np.mean(vrs)  if vrs  else np.nan,
            'iqr_ratio':  np.mean(irs)  if irs  else np.nan,
        })
    del X_layer_log
shape_df = pd.DataFrame(rows)

# ---- side-by-side table: every arm per marker ------------------------------
piv = shape_df.pivot(index='marker', columns='method',
                     values=['peak_ratio', 'var_ratio', 'iqr_ratio']).round(3)
piv = piv.reindex(columns=pd.MultiIndex.from_product(
    [['peak_ratio', 'var_ratio', 'iqr_ratio'], SHAPE_LABELS]))
piv = piv.sort_values(('peak_ratio', MAIN_LABEL))

print('=' * 92)
print('SHAPE-PRESERVATION DIAGNOSTIC  (1.0 = shape preserved; per-sample mean vs raw, log1p)')
print('  peak_ratio <1 = mode flattened   |   var_ratio / iqr_ratio >1 = broadened')
print('=' * 92)
print(piv.to_string())

by_method = {label: shape_df[shape_df.method == label].set_index('marker')
             for label in SHAPE_LABELS}
s1 = by_method[S1_LABEL]
PEAK_LO, VAR_HI, VAR_LO = 0.80, 1.25, 0.80

# ---- per-arm distortion flags ----------------------------------------------
print('\n' + '=' * 92)
print(f'SHAPE-DISTORTED markers per arm (peak<{PEAK_LO} or var outside [{VAR_LO},{VAR_HI}])')
print('=' * 92)
for label in SHAPE_LABELS[1:]:
    arm = by_method[label]
    distorted = arm[(arm.peak_ratio < PEAK_LO) |
                    (arm.var_ratio > VAR_HI) | (arm.var_ratio < VAR_LO)]
    print(f'\n{label}:  {len(distorted)} distorted')
    if len(distorted) == 0:
        print('  none — preserves 1D shape on all markers')
    else:
        for m, r in distorted.sort_values('peak_ratio').iterrows():
            tag = ' [bimodal/passed-through]' if m in bimodal_names else ''
            print(f'  {m:>10s}  peak={r.peak_ratio:.2f} (S1 {s1.loc[m,"peak_ratio"]:.2f})  '
                  f'var={r.var_ratio:.2f} (S1 {s1.loc[m,"var_ratio"]:.2f})  '
                  f'iqr={r.iqr_ratio:.2f}{tag}')

# ---- unimodal-only summary across arms -------------------------------------
uni = [m for m in s1.index if m not in bimodal_names]
uni_rows = []
for label in SHAPE_LABELS:
    arm = by_method[label]
    flagged = arm.loc[uni]
    uni_rows.append({
        'arm': label,
        'mean_peak_ratio': round(float(flagged['peak_ratio'].mean()), 3),
        'mean_var_ratio':  round(float(flagged['var_ratio'].mean()), 3),
        'mean_iqr_ratio':  round(float(flagged['iqr_ratio'].mean()), 3),
        'n_distorted': int(((flagged.peak_ratio < PEAK_LO) |
                            (flagged.var_ratio > VAR_HI) |
                            (flagged.var_ratio < VAR_LO)).sum()),
    })
print('\n' + '=' * 92)
print('Unimodal markers only (the ones Stage 2 actually corrects) — 1.0 = shape preserved')
print('=' * 92)
print(pd.DataFrame(uni_rows).to_string(index=False))
print(f'\n{S1_LABEL} is the control: a pure translation, so it should sit at ~1.0.')
print('An arm that matches it has not bought its kBET by reshaping marginals.')

# ---- bar charts: peak_ratio and var_ratio across arms ----------------------
order = list(piv.index)
x = np.arange(len(order))
n_m = len(SHAPE_LABELS)
w = 0.8 / n_m
palette_shape = ['steelblue', 'forestgreen', 'darkorange', 'mediumorchid', 'sienna']
fig, axes = plt.subplots(2, 1, figsize=(20, 11))
for ax, ratio, ref_lines in [
    (axes[0], 'peak_ratio', [1.0]),
    (axes[1], 'var_ratio',  [1.0, VAR_HI, VAR_LO]),
]:
    for i, label in enumerate(SHAPE_LABELS):
        offset = (i - (n_m - 1) / 2) * w
        ax.bar(x + offset, [by_method[label].loc[m, ratio] for m in order], w,
               label=label, color=palette_shape[i % len(palette_shape)], alpha=0.85)
    for rl in ref_lines:
        ax.axhline(rl, color='red', linestyle='--', alpha=0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{ratio}  (1.0 = preserved)')
    ax.set_ylabel(ratio)
    ax.legend(fontsize=8, ncol=n_m)
    ax.grid(True, alpha=0.3, axis='y')
plt.suptitle('Shape preservation per ablation arm (Stage 1 = pure-shift control)', fontsize=13)
plt.tight_layout()
plt.show()

print('\nReading it: Stage 1 bars hug 1.0 (translation preserves shape). Every arm should also')
print('stay near 1.0 — if a reduced arm departs, the dropped loss WAS protecting 1D shape.')

## 8. kBET + silhouette

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

# Stage 1 is deterministic (kBET 0.6202 in every run) -> skipped to save time.
EVAL_LAYERS = [k for k, _ in ABL_LAYERS if k != 'normalized_base']
print(f'Evaluating {len(EVAL_LAYERS)} layers: {[ABL_LABEL[k] for k in EVAL_LAYERS]}')

In [ ]:
import anndata as ad
import gc

all_kbet = {}
umap_coords = {}

# MEMORY NOTE: the original cell did `adata_kbet = adata_norm.copy()`, which duplicates
# EVERY layer (~141 MB each; 6 layers once the ablation arms exist = ~850 MB per
# iteration, 5 iterations). pegasus only ever reads .X here, so build a minimal
# AnnData instead: X + obs + var. That keeps the per-iteration cost at one 141 MB
# array rather than ~850 MB, and it is faster too.
for layer_name in EVAL_LAYERS:
    label = ABL_LABEL.get(layer_name, layer_name)
    print(f'\n{"-"*55}\n  {label}  ({layer_name})\n{"-"*55}')

    adata_kbet = ad.AnnData(
        X=np.asarray(adata_norm.layers[layer_name], dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}
        finally:
            del parts, adata_g, mmdata

    all_kbet[layer_name] = layer_res
    del adata_kbet
    gc.collect()

print('\nDone.')

In [ ]:
print('=' * 70)
print('kBET Summary (mean across groups — higher = better)')
print('=' * 70)
summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    if kbets:
        row = {'arm': ABL_LABEL.get(layer_name, layer_name), 'layer': layer_name,
               'mean_kBET': float(np.mean(kbets)), 'mean_chi2': float(np.mean(chi2s)),
               'n_groups': f'{len(kbets)}/{len(res)}'}
        summary_rows.append(row)
        print(f"  {row['arm']:>12s} ({layer_name:>15s})  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  ({row['n_groups']} groups)")
df_kbet = pd.DataFrame(summary_rows)

print('\n' + '=' * 70)
print('Per-group kBET')
print('=' * 70)
pergroup = pd.DataFrame({
    ABL_LABEL.get(layer_name, layer_name): {g: v['kBET'] for g, v in res.items()}
    for layer_name, res in all_kbet.items()
}).round(4)
print(pergroup.to_string())

if len(df_kbet):
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.bar(df_kbet['arm'], df_kbet['mean_kBET'], color='steelblue', alpha=0.85)
    ax.axhline(0.6315, color='red', linestyle='--', linewidth=1.3, label='UniFORM 0.6315')
    ax.axhline(0.6202, color='black', linestyle=':', linewidth=1.3, label='Stage 1 0.6202')
    ax.set_ylabel('Mean kBET acceptance rate')
    ax.set_title('kBET — GNN vs matched-noise controls')
    ax.set_xticks(range(len(df_kbet)))
    ax.set_xticklabels(df_kbet['arm'], rotation=20, ha='right', fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
# Silhouette score — UniFORM-style: 3 cell types, computed PER SAMPLE
# Tumor epithelial (ECAD+), immune (CD45+), non-immune stromal (aSMA+)
#
# Why per-sample (not per clinical group): computing silhouette across two
# batch-paired samples lets the cross-batch offset inflate the raw baseline —
# good normalization then looks like it *lowers* silhouette (it just removed the
# batch-driven separation). Computing within each sample removes that confound:
# the score reflects only whether the 3 cell populations stay distinct.
# Labels are assigned once on raw data, fixed across all methods.

N_SIL = 3000           # cells per sample; O(n²) — stable well below this
MIN_TYPE_CELLS = 50    # a cell type counts as "present" if it has >= this many cells
                        # (decided on FULL per-sample counts → deterministic, seed-independent)

# Step 1: GMM thresholds for 3 anchor markers on full raw dataset (log1p space)
anchor_names = ['ECAD', 'CD45', 'aSMA']
anchor_thresholds = {}
print('Anchor marker GMM thresholds (log1p, raw data):')
for mname in anchor_names:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(
        X_log_preview[:, mi].reshape(-1, 1)
    )
    thr = float(np.mean(gmm.means_))
    anchor_thresholds[mname] = thr
    print(f'  {mname}: {thr:.3f}')

# Step 2: assign cell type labels on raw data — fixed for all methods
#   0 = tumor epithelial (ECAD+)
#   1 = immune           (CD45+)
#   2 = stromal          (aSMA+)
#  -1 = unassigned
# Priority for conflicts: ECAD > CD45 > aSMA
ecad_pos = X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']
cd45_pos = X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']
asma_pos = X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[asma_pos] = 2
cell_type[cd45_pos] = 1
cell_type[ecad_pos] = 0

print('\nCell type assignments (raw data):')
for code, label in [(0, 'tumor epithelial (ECAD+)'),
                    (1, 'immune (CD45+)'),
                    (2, 'stromal (aSMA+)'),
                    (-1, 'unassigned')]:
    n = (cell_type == code).sum()
    print(f'  {label}: {n:,}  ({100*n/len(cell_type):.1f}%)')

# Step 3: silhouette per sample, per method (no cross-batch mixing)
# Guard (deterministic): a sample is kept iff >= 2 cell types each have
# >= MIN_TYPE_CELLS cells on FULL data. Only those well-populated types are
# carried into the silhouette; sparse types (e.g. near-absent immune) are
# excluded from the computation rather than letting a subsample lottery drop
# the whole sample. This makes membership seed-independent and reproducible
# across notebooks.
print(f'\nSilhouette score per sample — 3 cell types in 20D log1p space (higher = better):')
sample_ids_all = adata_norm.obs['sample_id'].values
sil_rows = []
dropped = []
for sample in sorted(np.unique(sample_ids_all)):
    smask = sample_ids_all == sample
    s_labels = cell_type[smask]

    # full-data per-type counts → which types are substantial
    full_counts = {c: int((s_labels == c).sum()) for c in (0, 1, 2)}
    big = [c for c, n in full_counts.items() if n >= MIN_TYPE_CELLS]
    if len(big) < 2:
        dropped.append(sample)
        continue  # need >= 2 substantial cell types to define a silhouette

    # keep only the substantial types, then subsample for the O(n²) silhouette
    valid = np.where(np.isin(s_labels, big))[0]
    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sel_labels = s_labels[sel]

    sil_by_layer = {}
    for layer_name in ['raw'] + [k for k, _ in BIO_LAYERS]:
        if layer_name == 'raw':
            Xs = X_raw[smask]
        else:
            Xs = np.asarray(adata_norm.layers[layer_name])[smask]
        X_log = np.log1p(np.clip(Xs[sel], 0, None))
        sil_by_layer[layer_name] = round(
            float(silhouette_score(X_log, sel_labels, metric='euclidean')), 4
        )
    sil_rows.append({'sample': sample, **sil_by_layer})

sil_df = pd.DataFrame(sil_rows).set_index('sample')
sil_df.loc['Mean'] = sil_df.mean().round(4)
print(sil_df.to_string())
print(f'\n({len(sil_rows)} samples passed the guard: >=2 cell types with >={MIN_TYPE_CELLS} cells)')
if dropped:
    print(f'(dropped {len(dropped)}: {", ".join(dropped)})')
print('(higher = better — tumor / immune / stromal populations stay distinct after normalization)')


## 9. Paired 3-seed result (paste this)

Seed 0 is hard-coded from `spancy_shift_ddpm_avg.ipynb` (2026-10-02). Shape numbers are means over
the unimodal markers; `dist` = unimodal markers flagged (peak < 0.80 or var outside [0.80, 1.25]).

In [ ]:
S1_SIL = float(sil_df.loc['Mean', 'normalized_base'])
SEED0 = {   # spancy_shift_ddpm_avg.ipynb, seed 0 (kBET 5 groups, silhouette 19 samples, shape unimodal)
    'GNN':         dict(kBET=0.7131, sil=0.3619, peak=1.009, var=0.955, iqr=0.991, dist=0),
    'S1+noise':    dict(kBET=0.6936, sil=0.3514, peak=0.918, var=1.054, iqr=1.072, dist=2),
    'S1+shuffled': dict(kBET=0.6929, sil=0.3520, peak=0.927, var=1.054, iqr=1.067, dist=0),
}

rows = [dict(seed=0, arm=a, **v) for a, v in SEED0.items()]
for s in SEEDS:
    for arm, key in [('GNN', f'gnn_s{s}'), ('S1+noise', f'gctrl_gauss_s{s}'),
                     ('S1+shuffled', f'gctrl_perm_s{s}')]:
        kv = [r['kBET'] for r in all_kbet[key].values() if not np.isnan(r['kBET'])]
        sh = shape_df[shape_df.method == ABL_LABEL[key]].set_index('marker').loc[uni_markers]
        rows.append(dict(
            seed=s, arm=arm,
            kBET=float(np.mean(kv)) if kv else np.nan,
            sil=float(sil_df.loc['Mean', key]),
            peak=float(sh.peak_ratio.mean()), var=float(sh.var_ratio.mean()),
            iqr=float(sh.iqr_ratio.mean()),
            dist=int(((sh.peak_ratio < 0.80) | (sh.var_ratio > 1.25) | (sh.var_ratio < 0.80)).sum()),
        ))
res = pd.DataFrame(rows)

print('=' * 88)
print(f'PER SEED  (Stage 1: kBET 0.6202, silhouette {S1_SIL:.4f}, shape 1.000/0.994/1.000, 0 dist)')
print('=' * 88)
print(res.sort_values(['seed', 'arm']).round(4).to_string(index=False))

print('\n' + '=' * 88)
print('MEAN +/- SD over 3 seeds')
print('=' * 88)
agg = res.groupby('arm')[['kBET', 'sil', 'peak', 'var', 'iqr', 'dist']].agg(['mean', 'std'])
print(agg.round(4).to_string())

print('\n' + '=' * 88)
print('PAIRED: GNN minus control, within each seed')
print('=' * 88)
piv = res.pivot(index='seed', columns='arm')
for ctrl in ['S1+noise', 'S1+shuffled']:
    for m, better in [('kBET', 'higher'), ('sil', 'higher')]:
        d = piv[(m, 'GNN')] - piv[(m, ctrl)]
        n_pos = int((d > 0).sum())
        print(f'  GNN - {ctrl:11s} {m:4s}: ' + '  '.join(f's{s} {v:+.4f}' for s, v in d.items())
              + f'   mean {d.mean():+.4f} +/- {d.std():.4f}   GNN {better} in {n_pos}/{len(d)} seeds')

print('\n' + '=' * 88)
sil_d = piv[('sil', 'GNN')] - piv[('sil', 'S1+noise')]
kb_d = piv[('kBET', 'GNN')] - piv[('kBET', 'S1+noise')]
if (sil_d > 0).all() and (kb_d > 0).all():
    print('CONFIRMED at 3 seeds: the GNN beats noise of the same size on BOTH kBET and silhouette')
    print('every seed -> its correction is structured, not blur.')
else:
    print('NOT consistent across seeds — report the per-seed numbers and soften the claim:')
    print(f'  kBET higher in {int((kb_d > 0).sum())}/3, silhouette higher in {int((sil_d > 0).sum())}/3 seeds.')